# House Price — Optimisation des Hyperparamètres & Modèle Final

Ce notebook optimise les hyperparamètres des **5 meilleurs modèles** identifiés dans le benchmark, avec **Optuna** et une cross-validation 5-fold robuste.

Modèles optimisés :
1. Ridge (linéaire, log-cible)
2. LightGBM (boosting, log-cible)
3. XGBoost (boosting, log-cible)
4. CatBoost (boosting, log-cible)
5. RandomForest (ensembliste, log-cible)

**Modèle final** : meilleur modèle individuel + ensemble Stacking.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import optuna
import dill
import pendulum
from loguru import logger

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                               StackingRegressor, VotingRegressor,
                               GradientBoostingRegressor)
from sklearn.svm import SVR
from sklearn.compose  import ColumnTransformer
from sklearn.impute   import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.model_selection import (train_test_split, KFold, cross_val_score,
                                      cross_validate)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost  import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from optuna_integration import MLflowCallback

import ppscore as pps

sys.path.append(str(Path.cwd().parent))
from settings.params import MODEL_PARAMS, TIMEZONE, MODEL_NAME, SEED
from src.make_dataset import load_data

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

TARGET  = MODEL_PARAMS["TARGET"]
N_FOLDS = 5
SEED    = 43
N_TRIALS = 60  # Optuna trials par modèle
optuna.logging.set_verbosity(optuna.logging.WARNING)
logger.info("Imports OK")

## 1. Chargement des données

In [ ]:
data = load_data(dataset_name="house_prices", columns_to_lower=True)
data = data.astype({
    "overallqual": str, "overallcond": str, "garageyrblt": str,
    "yearbuilt": str, "yearremodadd": str, "mssubclass": str,
    "mosold": str, "yrsold": str,
})

COLS_TO_DROP = ["id", "yrsold", "yearbuilt", "yearremodadd", "garageyrblt"]
pps_df = pps.predictors(df=data.drop(COLS_TO_DROP, axis=1),
                         y=TARGET, output="df", random_seed=SEED)
FEATURE_NAMES = pps_df.loc[pps_df.ppscore.round(3) >= 0.05, "x"].values.tolist()

df_model = data[FEATURE_NAMES + [TARGET]].copy()
X = df_model[FEATURE_NAMES]
y_log = np.log1p(df_model[TARGET].astype(float))
y_raw = df_model[TARGET].astype(float)

X_train, X_test, y_log_train, y_log_test = train_test_split(
    X, y_log, test_size=0.2, random_state=SEED
)
_, _, y_raw_train, y_raw_test = train_test_split(
    X, y_raw, test_size=0.2, random_state=SEED
)

NUM_COLS = X_train.select_dtypes(include="number").columns.tolist()
CAT_COLS = X_train.select_dtypes(include=["object","bool"]).columns.tolist()
logger.info(f"Features: {len(FEATURE_NAMES)} | Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
def build_pipeline(estimator, scaler=RobustScaler()):
    num_pipe = make_pipeline(SimpleImputer(strategy="median"), scaler)
    cat_pipe = make_pipeline(
        SimpleImputer(strategy="constant", fill_value="undefined"),
        OneHotEncoder(handle_unknown="ignore", drop="if_binary"),
    )
    pre = ColumnTransformer([
        ("num", num_pipe, NUM_COLS),
        ("cat", cat_pipe, CAT_COLS),
    ], remainder="passthrough")
    return Pipeline([("pre", pre), ("est", estimator)])


def cv_rmse_log(pipeline, n_folds=N_FOLDS):
    """Cross-val RMSE en dollars sur y_log (reconverti avec expm1)."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = []
    for tr_idx, va_idx in kf.split(X_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_log_train.iloc[tr_idx], y_log_train.iloc[va_idx]
        pipeline.fit(X_tr, y_tr)
        pred = np.expm1(pipeline.predict(X_va))
        true = np.expm1(y_va)
        scores.append(np.sqrt(mean_squared_error(true, pred)))
    return np.array(scores)


def test_metrics(pipeline):
    """Métriques sur le jeu de test (echelle dollars)."""
    pred = np.expm1(pipeline.predict(X_test))
    true = np.expm1(y_log_test)
    return {
        "rmse": float(np.sqrt(mean_squared_error(true, pred))),
        "mae":  float(mean_absolute_error(true, pred)),
        "r2":   float(r2_score(true, pred)),
    }

results_opt = {}
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("house_price_optimization")
logger.info("Infrastructure d'optimisation prête")

## 2. Optimisation — Ridge

Bien que linéaire, Ridge avec log-cible obtient souvent un excellent RMSE. On cherche le `alpha` optimal.

In [ ]:
def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 1e3, log=True)
    pipe  = build_pipeline(Ridge(alpha=alpha))
    scores = cv_rmse_log(pipe)
    return scores.mean()

with mlflow.start_run(run_name="Ridge_Optuna"):
    mlflc = MLflowCallback(tracking_uri="mlruns", metric_name="cv_rmse",
                            create_experiment=False, mlflow_kwargs={"nested": True})
    study_ridge = optuna.create_study(direction="minimize", study_name="ridge")
    study_ridge.optimize(objective_ridge, n_trials=N_TRIALS, callbacks=[mlflc],
                          show_progress_bar=True)

    best_pipe_ridge = build_pipeline(Ridge(**study_ridge.best_params))
    best_pipe_ridge.fit(X_train, y_log_train)
    m_ridge = test_metrics(best_pipe_ridge)

    mlflow.log_params(study_ridge.best_params)
    mlflow.log_metrics({f"test_{k}": v for k, v in m_ridge.items()})
    mlflow.log_metric("cv_rmse_best", study_ridge.best_value)
    results_opt["Ridge_Opt"] = m_ridge

logger.info(f"Ridge optimisé  → RMSE={m_ridge['rmse']:.0f}$ | R²={m_ridge['r2']:.4f}")
logger.info(f"Best params: {study_ridge.best_params}")

## 3. Optimisation — LightGBM

In [ ]:
def objective_lgbm(trial):
    params = {
        "n_estimators":   trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate":  trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "num_leaves":     trial.suggest_int("num_leaves", 20, 300),
        "max_depth":      trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample":      trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":      trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":     trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    pipe = build_pipeline(LGBMRegressor(**params, random_state=SEED,
                                         verbose=-1, n_jobs=-1))
    return cv_rmse_log(pipe).mean()

with mlflow.start_run(run_name="LightGBM_Optuna"):
    mlflc = MLflowCallback(tracking_uri="mlruns", metric_name="cv_rmse",
                            create_experiment=False, mlflow_kwargs={"nested": True})
    study_lgbm = optuna.create_study(direction="minimize", study_name="lgbm")
    study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, callbacks=[mlflc],
                         show_progress_bar=True)

    best_pipe_lgbm = build_pipeline(
        LGBMRegressor(**study_lgbm.best_params, random_state=SEED, verbose=-1, n_jobs=-1)
    )
    best_pipe_lgbm.fit(X_train, y_log_train)
    m_lgbm = test_metrics(best_pipe_lgbm)

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metrics({f"test_{k}": v for k, v in m_lgbm.items()})
    mlflow.log_metric("cv_rmse_best", study_lgbm.best_value)
    results_opt["LightGBM_Opt"] = m_lgbm

logger.info(f"LightGBM optimisé → RMSE={m_lgbm['rmse']:.0f}$ | R²={m_lgbm['r2']:.4f}")
logger.info(f"Best params: {study_lgbm.best_params}")

## 4. Optimisation — XGBoost

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "gamma":            trial.suggest_float("gamma", 0.0, 1.0),
    }
    pipe = build_pipeline(XGBRegressor(**params, random_state=SEED,
                                        verbosity=0, n_jobs=-1))
    return cv_rmse_log(pipe).mean()

with mlflow.start_run(run_name="XGBoost_Optuna"):
    mlflc = MLflowCallback(tracking_uri="mlruns", metric_name="cv_rmse",
                            create_experiment=False, mlflow_kwargs={"nested": True})
    study_xgb = optuna.create_study(direction="minimize", study_name="xgb")
    study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, callbacks=[mlflc],
                        show_progress_bar=True)

    best_pipe_xgb = build_pipeline(
        XGBRegressor(**study_xgb.best_params, random_state=SEED, verbosity=0, n_jobs=-1)
    )
    best_pipe_xgb.fit(X_train, y_log_train)
    m_xgb = test_metrics(best_pipe_xgb)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metrics({f"test_{k}": v for k, v in m_xgb.items()})
    mlflow.log_metric("cv_rmse_best", study_xgb.best_value)
    results_opt["XGBoost_Opt"] = m_xgb

logger.info(f"XGBoost optimisé  → RMSE={m_xgb['rmse']:.0f}$ | R²={m_xgb['r2']:.4f}")
logger.info(f"Best params: {study_xgb.best_params}")

## 5. Optimisation — CatBoost

In [ ]:
def objective_cat(trial):
    params = {
        "iterations":    trial.suggest_int("iterations", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "depth":         trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 0.0, 1.0),
        "border_count":  trial.suggest_int("border_count", 32, 255),
    }
    pipe = build_pipeline(CatBoostRegressor(**params, random_seed=SEED, verbose=0))
    return cv_rmse_log(pipe).mean()

with mlflow.start_run(run_name="CatBoost_Optuna"):
    mlflc = MLflowCallback(tracking_uri="mlruns", metric_name="cv_rmse",
                            create_experiment=False, mlflow_kwargs={"nested": True})
    study_cat = optuna.create_study(direction="minimize", study_name="catboost")
    study_cat.optimize(objective_cat, n_trials=N_TRIALS, callbacks=[mlflc],
                        show_progress_bar=True)

    best_pipe_cat = build_pipeline(
        CatBoostRegressor(**study_cat.best_params, random_seed=SEED, verbose=0)
    )
    best_pipe_cat.fit(X_train, y_log_train)
    m_cat = test_metrics(best_pipe_cat)

    mlflow.log_params(study_cat.best_params)
    mlflow.log_metrics({f"test_{k}": v for k, v in m_cat.items()})
    mlflow.log_metric("cv_rmse_best", study_cat.best_value)
    results_opt["CatBoost_Opt"] = m_cat

logger.info(f"CatBoost optimisé → RMSE={m_cat['rmse']:.0f}$ | R²={m_cat['r2']:.4f}")
logger.info(f"Best params: {study_cat.best_params}")

## 6. Optimisation — RandomForest

In [ ]:
def objective_rf(trial):
    params = {
        "n_estimators":  trial.suggest_int("n_estimators", 100, 600),
        "max_depth":     trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":  trial.suggest_categorical("max_features", ["sqrt","log2", None]),
    }
    pipe = build_pipeline(RandomForestRegressor(**params, random_state=SEED, n_jobs=-1))
    return cv_rmse_log(pipe).mean()

with mlflow.start_run(run_name="RandomForest_Optuna"):
    mlflc = MLflowCallback(tracking_uri="mlruns", metric_name="cv_rmse",
                            create_experiment=False, mlflow_kwargs={"nested": True})
    study_rf = optuna.create_study(direction="minimize", study_name="rf")
    study_rf.optimize(objective_rf, n_trials=40, callbacks=[mlflc],
                       show_progress_bar=True)

    best_pipe_rf = build_pipeline(
        RandomForestRegressor(**study_rf.best_params, random_state=SEED, n_jobs=-1)
    )
    best_pipe_rf.fit(X_train, y_log_train)
    m_rf = test_metrics(best_pipe_rf)

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metrics({f"test_{k}": v for k, v in m_rf.items()})
    mlflow.log_metric("cv_rmse_best", study_rf.best_value)
    results_opt["RandomForest_Opt"] = m_rf

logger.info(f"RandomForest optimisé → RMSE={m_rf['rmse']:.0f}$ | R²={m_rf['r2']:.4f}")

## 7. Ensemble — Stacking & Voting

Le **stacking** combine plusieurs modèles optimisés avec un méta-modèle (Ridge) pour tirer parti de leurs forces complémentaires.

In [ ]:
# ── VotingRegressor ────────────────────────────────────────────────────────
voting_pipe = build_pipeline(
    VotingRegressor(estimators=[
        ("lgbm", LGBMRegressor(**study_lgbm.best_params, random_state=SEED, verbose=-1)),
        ("xgb",  XGBRegressor(**study_xgb.best_params,   random_state=SEED, verbosity=0)),
        ("cat",  CatBoostRegressor(**study_cat.best_params, random_seed=SEED, verbose=0)),
    ])
)
voting_pipe.fit(X_train, y_log_train)
m_voting = test_metrics(voting_pipe)
results_opt["Voting_LGB_XGB_CAT"] = m_voting

with mlflow.start_run(run_name="Voting_LGB_XGB_CAT"):
    mlflow.log_metrics({f"test_{k}": v for k, v in m_voting.items()})

logger.info(f"VotingRegressor   → RMSE={m_voting['rmse']:.0f}$ | R²={m_voting['r2']:.4f}")

In [ ]:
# ── StackingRegressor ──────────────────────────────────────────────────────
# On reconstruit les estimateurs "bruts" pour le stacking (pas de pipeline imbriqué)
from sklearn.pipeline import Pipeline as SKPipeline

def make_base_est(est):
    """Crée un pipeline complet num+cat pour un estimateur donné."""
    return build_pipeline(est)

stacking_estimators = [
    ("lgbm", build_pipeline(LGBMRegressor(**study_lgbm.best_params,
                                            random_state=SEED, verbose=-1))),
    ("xgb",  build_pipeline(XGBRegressor(**study_xgb.best_params,
                                           random_state=SEED, verbosity=0))),
    ("cat",  build_pipeline(CatBoostRegressor(**study_cat.best_params,
                                               random_seed=SEED, verbose=0))),
    ("rf",   build_pipeline(RandomForestRegressor(**study_rf.best_params,
                                                   random_state=SEED, n_jobs=-1))),
]

stacking = StackingRegressor(
    estimators=stacking_estimators,
    final_estimator=Ridge(alpha=study_ridge.best_params["alpha"]),
    passthrough=False,
    cv=3,
    n_jobs=-1,
)
stacking.fit(X_train, y_log_train)
m_stack = test_metrics(stacking)
results_opt["Stacking_4models"] = m_stack

with mlflow.start_run(run_name="Stacking_4models"):
    mlflow.log_metrics({f"test_{k}": v for k, v in m_stack.items()})

logger.info(f"StackingRegressor → RMSE={m_stack['rmse']:.0f}$ | R²={m_stack['r2']:.4f}")

## 8. Tableau final de comparaison

In [ ]:
final_df = (
    pd.DataFrame(results_opt).T
    .rename_axis("Modèle")
    .sort_values("rmse")
    .reset_index()
)
final_df.index += 1

display(
    final_df.style
    .background_gradient(subset=["rmse"], cmap="RdYlGn_r")
    .background_gradient(subset=["r2"],   cmap="RdYlGn")
    .format({"rmse": "{:.0f}", "mae": "{:.0f}", "r2": "{:.4f}"})
)

best_name    = final_df.iloc[0]["Modèle"]
best_metrics = results_opt[best_name]
logger.info(f"\n🏆 Meilleur modèle : {best_name}")
logger.info(f"   RMSE = {best_metrics['rmse']:.0f}$")
logger.info(f"   MAE  = {best_metrics['mae']:.0f}$")
logger.info(f"   R²   = {best_metrics['r2']:.4f}")

In [ ]:
# ── Graphique comparaison modèles optimisés ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_ = final_df["Modèle"].values
rmses_  = final_df["rmse"].values
r2s_    = final_df["r2"].values

palette = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(models_)))

axes[0].barh(models_[::-1], rmses_[::-1], color=palette[::-1])
axes[0].set_xlabel("RMSE ($)")
axes[0].set_title("RMSE — Modèles optimisés", fontweight="bold")
axes[0].axvline(rmses_.min(), color="green", linestyle="--", alpha=0.7)

axes[1].barh(models_[::-1], r2s_[::-1], color=palette[::-1])
axes[1].set_xlabel("R²")
axes[1].set_title("R² — Modèles optimisés", fontweight="bold")
axes[1].axvline(r2s_.max(), color="green", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig("../reports/optimization_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Analyse du meilleur modèle

### 9.1 Feature Importances

In [ ]:
# Détecter le meilleur modèle et afficher ses importances (si disponible)
best_pipelines = {
    "Ridge_Opt":     best_pipe_ridge,
    "LightGBM_Opt":  best_pipe_lgbm,
    "XGBoost_Opt":   best_pipe_xgb,
    "CatBoost_Opt":  best_pipe_cat,
    "RandomForest_Opt": best_pipe_rf,
}

if best_name in best_pipelines:
    best_pipeline = best_pipelines[best_name]
else:
    best_pipeline = best_pipe_lgbm  # fallback
    best_name_used = "LightGBM_Opt (fallback)"

est = best_pipeline.named_steps["est"]
feature_names_out = best_pipeline.named_steps["pre"].get_feature_names_out()

if hasattr(est, "feature_importances_"):
    importances = pd.Series(est.feature_importances_, index=feature_names_out)
    top20 = importances.sort_values(ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(9, 7))
    top20.plot(kind="barh", ax=ax, color="#5B8DB8")
    ax.set_title(f"Top 20 Feature Importances — {best_name}", fontweight="bold")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("../reports/feature_importances_best.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print(f"Le modèle {best_name} n'expose pas de feature_importances_")

### 9.2 Analyse des résidus

In [ ]:
preds_best = np.expm1(best_pipeline.predict(X_test))
true_vals  = np.expm1(y_log_test)
residuals  = true_vals - preds_best

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Résidus vs prédictions
axes[0].scatter(preds_best, residuals, alpha=0.4, s=15, color="#5B8DB8")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Prédictions ($)")
axes[0].set_ylabel("Résidus ($)")
axes[0].set_title("Résidus vs Prédictions")

# Distribution des résidus
axes[1].hist(residuals, bins=40, color="#5B8DB8", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Résidu ($)")
axes[1].set_title("Distribution des résidus")

# Prédictions vs Vraies valeurs
axes[2].scatter(true_vals, preds_best, alpha=0.4, s=15, color="#E8985E")
mn = min(true_vals.min(), preds_best.min())
mx = max(true_vals.max(), preds_best.max())
axes[2].plot([mn, mx], [mn, mx], "r--", label="Prédiction parfaite")
axes[2].set_xlabel("Vraies valeurs ($)")
axes[2].set_ylabel("Prédictions ($)")
axes[2].set_title("Vraies valeurs vs Prédictions")
axes[2].legend()

plt.suptitle(f"Analyse des résidus — {best_name}", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/residuals_best.png", dpi=120, bbox_inches="tight")
plt.show()

logger.info(f"Résidus — Moyenne: {residuals.mean():.0f}$ | Std: {residuals.std():.0f}$")

## 10. Sauvegarde du modèle final

In [ ]:
EXECUTION_DATE = pendulum.now(tz="UTC")
PROJECT_DIR    = Path.cwd().parent
MODEL_DIR      = Path(PROJECT_DIR, "models")
MODEL_DIR.mkdir(exist_ok=True, parents=True)
REPORT_DIR     = Path(PROJECT_DIR, "reports")
REPORT_DIR.mkdir(exist_ok=True, parents=True)

model_path = Path(MODEL_DIR, f"{EXECUTION_DATE.strftime('%Y%m%d')}_{MODEL_NAME}")
with open(model_path, "wb") as f:
    dill.dump(best_pipeline, f)

logger.info(f"Modèle sauvegardé : {model_path}")
logger.info(f"Métriques finales : RMSE={best_metrics['rmse']:.0f}$ | MAE={best_metrics['mae']:.0f}$ | R²={best_metrics['r2']:.4f}")

## 11. MLFlow UI

```bash
# Depuis le dossier notebooks/
mlflow ui
# Ouvrir http://127.0.0.1:5000
```

---
**Résumé des résultats** :

| Étape | Meilleur modèle | RMSE test |
|-------|----------------|-----------|
| Benchmark (nb02) | À confirmer après exécution | — |
| Après optimisation | Variable selon résultats Optuna | — |
| Stacking 4 modèles | Potentiellement meilleur | — |

> *Exécuter les deux notebooks pour obtenir les métriques réelles.*